# Cryptographic Faithful Aggregation on Edge-IIoTset - v3 (15-class IDS, stronger attacks)

**What changed vs v2**
* **15-class attack detection** (macro-F1) instead of binary: each site sees only some attack types, so federation should clearly beat local-only training.
* **Richer features**: payload/URI/text columns are no longer thrown away (v2 coerced them to numbers and lost the signal); ports are one-hot encoded.
* **Attacks that actually hurt**: targeted loss-ascent adaptive attacker (also at 30% Byzantine), targeted label flip (attacks relabelled Normal), backdoor trigger (attack success rate is measured), plus sign-flip and class-rotation label flip.
* **New reference designs** for the cosine test: tolerant threshold (tau < 0) and a momentum-smoothed root reference.
* **Logging**: per-round and per-site acceptance, site distance from the root distribution (fairness check), paired statistics with confidence intervals.

**How to run:** same folder as `faithful_agg.py` (the updated one, with tolerant-tau support) and `DNN-EdgeIIoT-dataset.csv`. Run `PROFILE="tune"` first (checks the IDS actually learns and that federation beats local-only), then `"quick"`, then `"full"`. Runs are saved after every experiment and resume if interrupted (delete the results folder if you change the config). Upload `results_pack.zip` (aggregates only) when done.

In [ ]:
# ============================== 1. CONFIG ==============================
import os, sys, json, time, math, platform, zipfile, datetime, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DATA_PATH = os.environ.get("EDGE_DATA", "DNN-EdgeIIoT-dataset.csv")
CORE_DIR  = os.environ.get("CORE_DIR", ".")
PROFILE   = os.environ.get("PROFILE", "quick")                  # "tune" | "quick" | "full"
OUT_DIR   = Path(os.environ.get("OUT_DIR", "results_v3_" + PROFILE))

COMMON = dict(
    n_sites=10, n_byz=2, dirichlet_alpha=0.5,
    hidden=24, max_features=80, use_payload_features=True,
    lr=0.1, batch=128, local_steps=30,
    n_normal=100_000, n_per_attack=10_000,
    root_size=1500, test_frac=0.2,
    krum_f=3, trim_k=3,                       # defenders assume up to 3 of 10 attackers (fixed for all scenarios)
    kappa=None, mom_beta=0.9, arm_filter=None,
    backdoor_frac=0.5, trigger_k=3, trigger_val=4.0,
    taus_prev=[0.0], taus_root=[-0.3, 0.0, 0.2], taus_mom=[0.0, 0.2],
)
PROFILES = {
    "tune":  dict(seeds=[0], rounds=30, scenarios=["none"],
                  arm_filter=["fedavg", "local_only", "centralized", "faithful_root_t0.0"]),
    "quick": dict(seeds=[0, 1], rounds=12, scenarios=["none", "signflip", "targeted_flip", "backdoor", "adaptive"],
                  taus_prev=[0.0], taus_root=[0.0], taus_mom=[0.0], n_normal=20_000, n_per_attack=2_000),
    "full":  dict(seeds=[0, 1, 2, 3, 4], rounds=30,
                  scenarios=["none", "signflip", "labelflip", "targeted_flip", "backdoor", "adaptive", "adaptive_f30"]),
}
CFG = {**COMMON, **PROFILES[PROFILE]}
CFG.update(json.loads(os.environ.get("CFG_OVERRIDES", "{}")))

sys.path.insert(0, str(Path(CORE_DIR).resolve()))
import faithful_agg as fa
assert hasattr(fa, "ideal_accept_tolerant"), "faithful_agg.py is outdated: download the updated version (tolerant-tau support)"
OUT_DIR.mkdir(parents=True, exist_ok=True); (OUT_DIR / "figs").mkdir(exist_ok=True)
print("profile:", PROFILE, "| output:", OUT_DIR.resolve())
print(json.dumps(CFG, indent=1))

## 2. Data: chunked sampling, feature engineering, class mapping
Placeholder values (`0`, `0.0`, `nan`) in text columns mean "field absent". For text columns we keep: present-flag, log length, share of special characters, and signature flags (SQL / XSS / path-traversal / upload tokens). Identifier columns (IPs, timestamps, source ports) are dropped. Low-cardinality text columns are one-hot encoded; destination ports are one-hot (top 15) plus a high-port flag. The top `max_features` features by strongest one-vs-rest correlation with any class are kept.

In [ ]:
# ============================== 2. DATA ==============================
IDENT_DROP = {"frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
              "icmp.transmit_timestamp", "tcp.srcport"}
PORT_COLS = ["tcp.dstport", "udp.port"]
PLACEHOLDERS = {"0", "0.0", "nan", "NaN", "", "None", "nan.0"}
RX = {"sql": r"(?i)(select|union|insert|drop|update|delete|or\s+1=1|sleep\(|--|%27)",
      "xss": r"(?i)(<script|javascript:|onerror|onload|alert\(|%3cscript|<img|<svg)",
      "trav": r"(?i)(\.\./|%2e%2e|/etc/passwd|cmd\.exe)",
      "upl": r"(?i)(multipart|filename=|\.php|\.jsp|\.exe|content-disposition)"}

def find_col(cols, name):
    for c in cols:
        if c.strip().lower() == name.lower():
            return c
    return None

head = pd.read_csv(DATA_PATH, nrows=5, low_memory=False)
YCOL, TCOL = find_col(head.columns, "Attack_label"), find_col(head.columns, "Attack_type")
assert YCOL and TCOL, f"label columns not found; last columns: {list(head.columns[-4:])}"
lab = pd.read_csv(DATA_PATH, usecols=[YCOL, TCOL], low_memory=False)
lab[TCOL] = lab[TCOL].astype(str)
counts = lab.groupby(TCOL)[YCOL].agg(["size", "mean"])
normal_types = set(counts.index[counts["mean"] == 0])
cap = {t: (CFG["n_normal"] if t in normal_types else CFG["n_per_attack"]) for t in counts.index}
keep_p = {t: (1.0 if cap[t] is None else min(1.0, cap[t] / counts.loc[t, "size"])) for t in counts.index}
FILE_ROWS, FILE_COUNTS = int(len(lab)), counts["size"].astype(int).to_dict()
print(f"{FILE_ROWS:,} rows in file"); print(counts["size"].to_string()); del lab

rng_s = np.random.default_rng(12345); parts = []
for ch in pd.read_csv(DATA_PATH, chunksize=250_000, low_memory=False):
    p = ch[TCOL].astype(str).map(keep_p).fillna(0).values
    parts.append(ch[rng_s.random(len(ch)) < p])
raw = pd.concat(parts, ignore_index=True); del parts

types_arr = raw[TCOL].astype(str).values
types_arr = np.where(np.isin(types_arr, list(normal_types)), "Normal", types_arr)
CLASSES = sorted(np.unique(types_arr)); C = len(CLASSES)
NORMAL = CLASSES.index("Normal")
y_arr = np.array([CLASSES.index(t) for t in types_arr])
y_bin = (y_arr != NORMAL).astype(int)

feats, skipped, groups = {}, [], {"numeric": 0, "onehot": 0, "text": 0, "port": 0}
for c in [c for c in raw.columns if c not in (YCOL, TCOL) and c.strip() not in IDENT_DROP and c not in PORT_COLS]:
    try:
        s = raw[c]
        if pd.api.types.is_numeric_dtype(s):
            feats[c] = s.astype(np.float32); groups["numeric"] += 1; continue
        st = s.astype(str); pres = ~st.isin(PLACEHOLDERS); num = pd.to_numeric(st, errors="coerce")
        if float((pres & num.isna()).mean()) < 0.001:
            feats[c] = num.fillna(0).astype(np.float32); groups["numeric"] += 1; continue
        feats[c + "__present"] = pres.astype(np.float32); groups["text"] += 1
        if st.nunique() <= 20:
            for name, col in pd.get_dummies(st, prefix=c, dtype=np.float32).items():
                feats[name] = col; groups["onehot"] += 1
        if CFG["use_payload_features"]:
            L = st.str.len().where(pres, 0)
            feats[c + "__loglen"] = np.log1p(L).astype(np.float32)
            feats[c + "__special"] = (st.str.count(r"[^A-Za-z0-9 ]").where(pres, 0) / L.clip(lower=1)).astype(np.float32)
            for nm, rx in RX.items():
                feats[f"{c}__{nm}"] = (st.str.contains(rx, regex=True) & pres).astype(np.float32)
    except Exception as e:
        skipped.append((c, repr(e)[:80]))
for c in PORT_COLS:
    if c in raw.columns:
        try:
            v = pd.to_numeric(raw[c].astype(str), errors="coerce").fillna(0)
            for pv in v.value_counts().index[:15]:
                feats[f"{c}=={int(pv)}"] = (v == pv).astype(np.float32); groups["port"] += 1
            feats[c + "__high"] = (v >= 1024).astype(np.float32)
        except Exception as e:
            skipped.append((c, repr(e)[:80]))
Xdf = pd.DataFrame(feats).apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)
Xdf = Xdf.loc[:, Xdf.std() > 0]
del raw, feats

Xv = Xdf.values; Xz = (Xv - Xv.mean(0)) / (Xv.std(0) + 1e-6)
Yc = np.eye(C, dtype=np.float32)[y_arr]; Yc = (Yc - Yc.mean(0)) / (Yc.std(0) + 1e-6)
score = (np.abs(Xz.T @ Yc) / len(Xz)).max(1)
order_f = np.argsort(-score)[: CFG["max_features"]]
FEATURES = [Xdf.columns[i] for i in order_f]
X_all = Xv[:, order_f].astype(np.float32); del Xz, Xdf, Xv
print(f"\nclasses ({C}): {CLASSES}")
print(f"sampled rows {len(y_arr):,} | attack share {y_bin.mean():.1%} | features kept {len(FEATURES)} (groups before selection: {groups})")
if skipped: print("skipped columns:", skipped)
print("top features:", FEATURES[:15])

## 3. Model, attacks, aggregators
Softmax MLP (NumPy, one flat parameter vector). Public norm bound each round `B_t = kappa * ||root update||`. The root set is a small clean server-side capture (e.g. a testbed). Attacks (2 of 10 sites, 3 for `*_f30`):
* `signflip`: -4u. `labelflip`: labels rotated y -> y+1. `targeted_flip`: all attack classes relabelled Normal (evasion). `backdoor`: 50% of attack samples get a trigger on 3 low-importance features and are relabelled Normal; **ASR** = share of triggered attack samples classified Normal.
* `adaptive`: worst case. Attacker knows B, tau and the reference; submits norm 0.95B at the minimum allowed cosine to the reference, the rest pointing along the **loss-ascent direction** (-honest update).

In [ ]:
# ============================== 3. MODEL / ATTACKS / AGGREGATORS ==============================
F = X_all.shape[1]; H = CFG["hidden"]; D = F * H + H + H * C + C
TRIG = np.arange(F - CFG["trigger_k"], F)          # lowest-importance features (selection order is by score)
print(f"input features {F}, classes {C}, model dimension d = {D}")

def unpack(w):
    i = 0
    W1 = w[i:i + F * H].reshape(F, H); i += F * H
    b1 = w[i:i + H]; i += H
    W2 = w[i:i + H * C].reshape(H, C); i += H * C
    return W1, b1, W2, w[i:i + C]

def init_w(rng):
    return np.concatenate([rng.normal(0, np.sqrt(2 / F), (F, H)).ravel(), np.zeros(H),
                           rng.normal(0, np.sqrt(2 / H), (H, C)).ravel(), np.zeros(C)])

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def predict_class(w, Xm):
    W1, b1, W2, b2 = unpack(w)
    return (np.maximum(Xm @ W1 + b1, 0.0) @ W2 + b2).argmax(1)

def local_update(w, Xm, ym, rng, batch=None):
    w0 = w.copy(); w = w.copy(); n = len(ym); bs = batch or CFG["batch"]
    for _ in range(CFG["local_steps"]):
        idx = rng.integers(0, n, bs)
        xb = Xm[idx].astype(np.float64); yb = ym[idx]
        W1, b1, W2, b2 = unpack(w)
        z1 = xb @ W1 + b1; a1 = np.maximum(z1, 0.0)
        P = softmax(a1 @ W2 + b2); P[np.arange(len(yb)), yb] -= 1.0; P /= len(yb)
        dz1 = (P @ W2.T) * (z1 > 0)
        g = np.concatenate([(xb.T @ dz1).ravel(), dz1.sum(0), (a1.T @ P).ravel(), P.sum(0)])
        w -= CFG["lr"] * g
    return w - w0

def metrics(w, Xm, ym, full=False):
    pred = predict_class(w, Xm)
    cm = np.bincount(ym * C + pred, minlength=C * C).reshape(C, C).astype(float)
    tp = np.diag(cm); sup = cm.sum(1); ps = cm.sum(0)
    rec = tp / np.maximum(sup, 1); prec = tp / np.maximum(ps, 1)
    f1c = 2 * prec * rec / np.maximum(prec + rec, 1e-12); pres = sup > 0
    att = ym != NORMAL; pa = pred != NORMAL
    out = dict(acc=float((pred == ym).mean()), macro_f1=float(f1c[pres].mean()), bal_acc=float(rec[pres].mean()),
               detect=float((pa & att).sum() / max(att.sum(), 1)), fpr=float((pa & ~att).sum() / max((~att).sum(), 1)))
    if full:
        out["per_class_recall"] = {CLASSES[i]: round(float(rec[i]), 3) for i in range(C) if pres[i]}
    return out

def asr(w, Xtrig):
    return float((predict_class(w, Xtrig) == NORMAL).mean()) if len(Xtrig) else float("nan")

def agg_trimmed(U, k):
    return np.sort(U, axis=0)[k:len(U) - k].mean(0) if (k > 0 and len(U) > 2 * k) else U.mean(0)

def agg_multikrum(U, f):
    n = len(U); G = U @ U.T; sq = np.diag(G); D2 = sq[:, None] + sq[None, :] - 2 * G
    mc = max(1, n - f - 2)
    scores = [np.sort(np.delete(D2[i], i))[:mc].sum() for i in range(n)]
    sel = np.argsort(scores)[:max(1, n - f)]
    return U[sel].mean(0), set(int(s) for s in sel)

def adaptive_vec(B, ref, tau, u_honest):
    neg = -u_honest
    if ref is None or np.linalg.norm(ref) == 0:
        return 0.95 * B * neg / (np.linalg.norm(neg) + 1e-12)
    rh = ref / np.linalg.norm(ref)
    p = neg - (neg @ rh) * rh; p /= (np.linalg.norm(p) + 1e-12)
    c = min(max(tau if tau is not None else 0.0, 0.0) + 0.02, 0.999)
    return 0.95 * B * (c * rh + np.sqrt(1 - c * c) * p)

def dirichlet_partition(train_idx, rng, tries=300, min_size=300):
    classes = np.unique(types_arr[train_idx])
    for _ in range(tries):
        parts = [[] for _ in range(CFG["n_sites"])]
        for t in classes:
            idx = rng.permutation(train_idx[types_arr[train_idx] == t])
            pr = rng.dirichlet([CFG["dirichlet_alpha"]] * CFG["n_sites"])
            cuts = (np.cumsum(pr) * len(idx)).astype(int)[:-1]
            for s, chunk in enumerate(np.split(idx, cuts)):
                parts[s].extend(chunk.tolist())
        if all(len(p) >= min_size for p in parts) and all(0.05 <= y_bin[p].mean() <= 0.95 for p in parts):
            return [np.array(p) for p in parts]
    raise RuntimeError("partition constraints not met: raise dirichlet_alpha / lower min_size / use more rows")

def build_bundle(seed):
    rng = np.random.default_rng(1000 + seed); test, root, train = [], [], []
    for t in np.unique(types_arr):
        idx = rng.permutation(np.where(types_arr == t)[0])
        nt = max(1, int(round(len(idx) * CFG["test_frac"]))); test.append(idx[:nt]); rest = idx[nt:]
        nr = max(1, int(round(CFG["root_size"] * len(idx) / len(types_arr)))); root.append(rest[:nr]); train.append(rest[nr:])
    test, root, train = (np.concatenate(a) for a in (test, root, train))
    mu = X_all[train].mean(0); sd = X_all[train].std(0) + 1e-6
    Z = np.clip((X_all - mu) / sd, -5, 5).astype(np.float32)
    parts = dirichlet_partition(train, rng)
    rh = np.bincount(y_arr[root], minlength=C) / len(root)
    tv = [float(0.5 * np.abs(np.bincount(y_arr[p], minlength=C) / len(p) - rh).sum()) for p in parts]
    Xt, yt = Z[test], y_arr[test]
    Xtrig = Xt[yt != NORMAL].copy(); Xtrig[:, TRIG] = CFG["trigger_val"]
    return dict(sites=[(Z[p], y_arr[p]) for p in parts], root=(Z[root], y_arr[root]), test=(Xt, yt), test_trig=Xtrig,
                site_sizes=[len(p) for p in parts], site_attack_frac=[float(y_bin[p].mean()) for p in parts],
                site_n_classes=[int(len(np.unique(y_arr[p]))) for p in parts], site_tv=tv)

In [ ]:
# ---- auto-calibrate kappa on CLEAN rounds (seed 0) ----
if CFG["kappa"] is None:
    b0 = build_bundle(0); rng = np.random.default_rng(99); w = init_w(rng); ratios = []
    for r in range(6):
        ru = local_update(w, *b0["root"], rng); rn = np.linalg.norm(ru)
        ups = [local_update(w, Xs, ys, rng) for Xs, ys in b0["sites"]]
        ratios += [np.linalg.norm(u) / rn for u in ups]; w = w + np.mean(ups, axis=0)
    p50, p99, pmax = np.percentile(ratios, [50, 99, 100])
    KAPPA = max(2.0, math.ceil(1.25 * p99 * 2) / 2)
    print(f"honest ||u||/||root||: median {p50:.2f}, p99 {p99:.2f}, max {pmax:.2f} -> kappa = {KAPPA}"); del b0
else:
    KAPPA = float(CFG["kappa"])
CFG["kappa_used"] = KAPPA
json.dump(CFG, open(OUT_DIR / "config.json", "w"), indent=2)

## 4. Experiment loop (resumable)
Arms: FedAvg, trimmed mean, multi-Krum, secure norm-only, Faithful with the previous-aggregate reference (shows capture), Faithful with the clean root reference (tau sweep including tolerant tau < 0), and Faithful with a momentum-smoothed root reference. Plus local-only and centralized reference rows. Secure arms run the real secret-sharing and Beaver-triple tests; commitments are measured separately in section 6.

In [ ]:
# ============================== 4. EXPERIMENT LOOP ==============================
ARMS = [("fedavg", "fedavg", None), ("trimmed_mean", "trimmed", None), ("multi_krum", "krum", None),
        ("secure_norm_only", "norm_only", None)]
ARMS += [(f"faithful_prev_t{t}", "faithful_prev", t) for t in CFG["taus_prev"]]
ARMS += [(f"faithful_root_t{t}", "faithful_root", t) for t in CFG["taus_root"]]
ARMS += [(f"faithful_rootmom_t{t}", "faithful_rootmom", t) for t in CFG["taus_mom"]]
if CFG["arm_filter"]:
    ARMS = [a for a in ARMS if a[0] in CFG["arm_filter"]]
RES_PATH = OUT_DIR / "results.csv"
J = json.dumps

def final_row(arm, fam, tau, scn, seed, n_byz, w, bundle, curve, extra, t_start):
    Xt, yt = bundle["test"]; m = metrics(w, Xt, yt, full=True)
    return dict(arm=arm, family=fam, tau=tau, scenario=scn, seed=seed, n_byz=n_byz, rounds=CFG["rounds"],
                final_macro_f1=m["macro_f1"], final_bal_acc=m["bal_acc"], final_acc=m["acc"], final_detect=m["detect"],
                final_fpr=m["fpr"], final_asr=asr(w, bundle["test_trig"]),
                last3_macro_f1=float(np.mean(curve[-3:])) if curve else m["macro_f1"],
                per_class_recall=J(m["per_class_recall"]), curve_macro_f1=J(curve),
                site_tv=J([round(x, 4) for x in bundle["site_tv"]]), seconds=round(time.time() - t_start, 1), **extra)

NOEXTRA = dict(honest_rejected=np.nan, honest_total=np.nan, byz_rejected=np.nan, byz_total=np.nan,
               hrej_curve="[]", site_accept="[]")

def run_one(arm, fam, tau, scn, seed, bundle):
    t_start = time.time()
    rng = np.random.default_rng(10_000 + seed); crng = np.random.default_rng(20_000 + seed)
    sites, (Xr, yr), (Xt, yt) = bundle["sites"], bundle["root"], bundle["test"]
    n = len(sites); rounds = CFG["rounds"]
    n_byz = 0 if scn == "none" else (3 if scn.endswith("_f30") else CFG["n_byz"]); base = scn.replace("_f30", "")
    byz = set(range(n_byz)); pois = {}
    for k in byz:
        Xs, ys = sites[k]
        if base == "labelflip": pois[k] = (Xs, (ys + 1) % C)
        elif base == "targeted_flip": pois[k] = (Xs, np.where(ys != NORMAL, NORMAL, ys))
        elif base == "backdoor":
            m = (rng.random(len(ys)) < CFG["backdoor_frac"]) & (ys != NORMAL)
            Xp = Xs.copy(); Xp[np.ix_(np.where(m)[0], TRIG)] = CFG["trigger_val"]; yp = ys.copy(); yp[m] = NORMAL
            pois[k] = (Xp, yp)
    w = init_w(rng); prev = None; mom = None
    acc_site = np.zeros(n); hrej_c = []; hr = hb = ht = bt = 0; curve = []
    for r in range(rounds):
        root_u = local_update(w, Xr, yr, rng); B = KAPPA * np.linalg.norm(root_u)
        mom = root_u if mom is None else CFG["mom_beta"] * mom + (1 - CFG["mom_beta"]) * root_u
        ref = {"faithful_root": root_u, "faithful_rootmom": mom, "faithful_prev": prev}.get(fam)
        ups = []
        for k, (Xs, ys) in enumerate(sites):
            if k in pois: u = local_update(w, *pois[k], rng)
            else: u = local_update(w, Xs, ys, rng)
            if k in byz:
                if base == "signflip": u = -4.0 * u
                elif base == "adaptive": u = adaptive_vec(B, ref, tau, u)
            ups.append(u)
        U = np.array(ups); mask = None
        if fam == "fedavg": agg = U.mean(0)
        elif fam == "trimmed": agg = agg_trimmed(U, CFG["trim_k"])
        elif fam == "krum":
            agg, sel = agg_multikrum(U, CFG["krum_f"]); mask = [i in sel for i in range(n)]
        else:
            subs = [fa.make_submission(k, u, crng, with_commitments=False) for k, u in enumerate(ups)]
            res = fa.run_round(subs, crng, B, tau, ref, verify_openings=False)
            agg = res.mean_update; mask = [bool(b) for b in res.bits]
            if res.n_accepted: prev = res.mean_update
        if mask is not None:
            h_r = 0
            for k in range(n):
                acc_site[k] += mask[k]
                if k in byz: bt += 1; hb += (not mask[k])
                else: ht += 1; hr += (not mask[k]); h_r += (not mask[k])
            hrej_c.append(h_r)
        w = w + agg
        curve.append(round(metrics(w, Xt, yt)["macro_f1"], 4))
    extra = NOEXTRA if mask is None else dict(honest_rejected=hr, honest_total=ht, byz_rejected=hb, byz_total=bt,
                                              hrej_curve=J(hrej_c), site_accept=J([int(x) for x in acc_site]))
    return final_row(arm, fam, tau, scn, seed, n_byz, w, bundle, curve, extra, t_start)

def run_local_only(seed, bundle):
    t0 = time.time(); rng = np.random.default_rng(30_000 + seed); Xt, yt = bundle["test"]; ms = []; ws = []
    for Xs, ys in bundle["sites"]:
        w = init_w(rng)
        for r in range(CFG["rounds"]): w = w + local_update(w, Xs, ys, rng)
        ms.append(metrics(w, Xt, yt)); ws.append(w)
    row = final_row("local_only", "local_only", None, "none", seed, 0, ws[0], bundle, [], NOEXTRA, t0)
    for k in ("macro_f1", "bal_acc", "acc", "detect", "fpr"):
        row["final_" + k] = float(np.mean([m[k] for m in ms]))
    row["last3_macro_f1"] = row["final_macro_f1"]; row["final_asr"] = float(np.mean([asr(w, bundle["test_trig"]) for w in ws]))
    return row

def run_centralized(seed, bundle):
    t0 = time.time(); rng = np.random.default_rng(40_000 + seed)
    Xp = np.concatenate([s[0] for s in bundle["sites"]]); yp = np.concatenate([s[1] for s in bundle["sites"]])
    w = init_w(rng); curve = []
    for r in range(CFG["rounds"]):
        w = w + local_update(w, Xp, yp, rng, batch=CFG["batch"] * CFG["n_sites"])
        curve.append(round(metrics(w, *bundle["test"])["macro_f1"], 4))
    return final_row("centralized", "centralized", None, "none", seed, 0, w, bundle, curve, NOEXTRA, t0)

done = set()
if RES_PATH.exists():
    old = pd.read_csv(RES_PATH); done = set(zip(old.arm, old.scenario, old.seed)); print(f"resuming: {len(done)} runs done")
want = lambda name: (not CFG["arm_filter"]) or (name in CFG["arm_filter"])
bundle_info = {}; t0 = time.time(); count = 0
for seed in CFG["seeds"]:
    bundle = build_bundle(seed)
    bundle_info[seed] = {k: bundle[k] for k in ("site_sizes", "site_attack_frac", "site_n_classes", "site_tv")}
    jobs = [("local_only", "local_only", None, "none")] * want("local_only") + [("centralized", "centralized", None, "none")] * want("centralized")
    jobs += [(a, f, t, s) for (a, f, t) in ARMS for s in CFG["scenarios"]]
    for arm, fam, tau, scn in jobs:
        count += 1
        if (arm, scn, seed) in done: continue
        row = (run_local_only(seed, bundle) if fam == "local_only" else run_centralized(seed, bundle) if fam == "centralized"
               else run_one(arm, fam, tau, scn, seed, bundle))
        pd.DataFrame([row]).to_csv(RES_PATH, mode="a", header=not RES_PATH.exists(), index=False)
        print(f"[{count}] seed={seed} {scn:13s} {arm:24s} macroF1={row['final_macro_f1']:.3f} detect={row['final_detect']:.3f} "
              f"fpr={row['final_fpr']:.3f} ASR={row['final_asr']:.3f} ({time.time() - t0:.0f}s)")
    del bundle
print("done.")

## 5. Summary, paired statistics, fairness, plots

In [ ]:
# ============================== 5. SUMMARY ==============================
import matplotlib.pyplot as plt
res = pd.read_csv(RES_PATH)
res["honest_rej_rate"] = res.honest_rejected / res.honest_total.replace(0, np.nan)
res["byz_rej_rate"] = res.byz_rejected / res.byz_total.replace(0, np.nan)
ci = lambda x: 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
summary = res.groupby(["scenario", "arm"]).agg(n=("seed", "count"), macro_f1=("final_macro_f1", "mean"), f1_ci95=("final_macro_f1", ci),
        bal_acc=("final_bal_acc", "mean"), detect=("final_detect", "mean"), fpr=("final_fpr", "mean"), asr=("final_asr", "mean"),
        honest_rej=("honest_rej_rate", "mean"), byz_rej=("byz_rej_rate", "mean")).reset_index()
summary.to_csv(OUT_DIR / "summary.csv", index=False)
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 500)
print(summary.round(3).to_string(index=False))

# paired differences (same seed = same partition and test set)
def paired(a, b, scn, col="final_macro_f1"):
    x = res[(res.arm == a) & (res.scenario == scn)].set_index("seed")[col]; y = res[(res.arm == b) & (res.scenario == scn)].set_index("seed")[col]
    d = (x - y).dropna(); n = len(d)
    return dict(A=a, B=b, scenario=scn, metric=col, mean_diff=d.mean(), ci95=1.96 * d.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan, wins=int((d > 0).sum()), n=n)
rows = []
fam_root = [a for a, f, t in ARMS if f in ("faithful_root", "faithful_rootmom", "faithful_prev")]
for scn in CFG["scenarios"]:
    for a in fam_root:
        for b in ("fedavg", "secure_norm_only", "multi_krum"):
            if a in set(res.arm) and b in set(res.arm):
                rows.append(paired(a, b, scn)); 
                if scn == "backdoor": rows.append(paired(a, b, scn, "final_asr"))
                if scn == "targeted_flip": rows.append(paired(a, b, scn, "final_detect"))
pairs = pd.DataFrame(rows); pairs.to_csv(OUT_DIR / "paired.csv", index=False)
print("\nPAIRED (A - B), key rows:"); print(pairs[pairs.A.str.contains("root_t0.0") | pairs.A.str.contains("prev")].round(3).to_string(index=False))

# fairness: per honest site acceptance vs distance from the root distribution (no-attack runs)
fr = []
for _, r in res[(res.scenario == "none") & (res.site_accept != "[]")].iterrows():
    acc_ = json.loads(r.site_accept); tv = json.loads(r.site_tv)
    for k in range(len(acc_)): fr.append(dict(arm=r.arm, seed=r.seed, site=k, accept_rate=acc_[k] / r.rounds, tv=tv[k]))
fair = pd.DataFrame(fr)
if len(fair):
    ft = fair.groupby("arm").apply(lambda g: pd.Series(dict(spearman_accept_vs_tv=g.accept_rate.corr(g.tv, method="spearman"),
            accept_lowTV=g[g.tv <= g.tv.quantile(1 / 3)].accept_rate.mean(), accept_highTV=g[g.tv >= g.tv.quantile(2 / 3)].accept_rate.mean()))).reset_index()
    ft.to_csv(OUT_DIR / "fairness.csv", index=False); print("\nFAIRNESS (no attack): does acceptance fall for sites far from the root distribution?"); print(ft.round(3).to_string(index=False))

order = ["centralized", "local_only"] + [a for a, _, _ in ARMS]
scns = CFG["scenarios"]
fig, axes = plt.subplots(1, len(scns), figsize=(4.0 * len(scns), 6), sharex=True); axes = np.atleast_1d(axes)
for ax, scn in zip(axes, scns):
    s = summary[summary.scenario == scn].set_index("arm"); s = s.reindex([a for a in order if a in s.index])
    ax.barh(s.index, s.macro_f1, xerr=s.f1_ci95.fillna(0), color="#4C78A8"); ax.set_title(scn); ax.set_xlim(0, 1); ax.set_xlabel("macro-F1")
plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "macro_f1_by_arm.png", dpi=130); plt.show()

sub = summary[summary.scenario.isin([s for s in ("backdoor", "targeted_flip") if s in scns])]
if len(sub):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, (scn, col, ttl) in zip(axes, [("backdoor", "asr", "backdoor attack success rate (lower is better)"), ("targeted_flip", "detect", "attack detection rate (higher is better)")]):
        s = summary[summary.scenario == scn].set_index("arm"); s = s.reindex([a for a in order if a in s.index])
        ax.barh(s.index, s[col], color="#E45756"); ax.set_title(ttl); ax.set_xlim(0, 1)
    plt.tight_layout(); plt.savefig(OUT_DIR / "figs" / "targeted_attacks.png", dpi=130); plt.show()

plt.figure(figsize=(7, 4.5))
for arm in [a for a in ("faithful_prev_t0.0", "faithful_root_t0.0", "faithful_rootmom_t0.0") if a in set(res.arm)]:
    sub = res[(res.arm == arm) & (res.scenario == "none")]
    if len(sub):
        cur = np.array([json.loads(s) for s in sub.hrej_curve]) / CFG["n_sites"]; plt.plot(cur.mean(0), label=arm)
plt.xlabel("round"); plt.ylabel("honest updates rejected (no attack)"); plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / "figs" / "honest_rejection_by_round.png", dpi=130); plt.show()

## 6. Cost benchmark with REAL commitments, Merkle inclusion and audit

In [ ]:
# ============================== 6. COST BENCHMARK ==============================
b0 = build_bundle(0); rb = np.random.default_rng(6); rc = np.random.default_rng(5)
w0 = init_w(rb); ups = [local_update(w0, Xs, ys, rb) for Xs, ys in b0["sites"]]
root_u = local_update(w0, *b0["root"], rb); Bq = KAPPA * np.linalg.norm(root_u)
t = time.perf_counter(); subs = [fa.make_submission(k, u, rc, with_commitments=True) for k, u in enumerate(ups)]; t_client = time.perf_counter() - t
led = fa.Ledger(); t = time.perf_counter()
res_c = fa.run_round(subs, rc, Bq, tau=0.2, ref=root_u, verify_openings=True, ledger=led); t_round = time.perf_counter() - t
t = time.perf_counter(); audit_ok = fa.audit(res_c.transcript); t_audit = time.perf_counter() - t
incl_ok = all(fa.site_verify(res_c.transcript, i, j)[0] for i in range(len(subs)) for j in range(2))
n = len(subs); placeholder_cmp = n * 3 * 2 * fa.CMP_ELEMS * fa.FE_BYTES; plain_bytes = n * D * 4
cost = dict(model_dim=D, n_sites=n, audit_ok=bool(audit_ok), inclusion_proofs_ok=bool(incl_ok), accepted=int(res_c.n_accepted),
            bytes_by_phase={k: int(v) for k, v in led.bytes_.items()}, secs_by_phase={k: round(v, 3) for k, v in led.secs.items()},
            client_commit_and_share_secs_total=round(t_client, 3), server_round_secs=round(t_round, 3), audit_secs=round(t_audit, 3),
            total_online_bytes=int(led.total_bytes()), placeholder_comparison_bytes=int(placeholder_cmp),
            online_bytes_excluding_placeholder=int(led.total_bytes() - placeholder_cmp),
            plain_fedavg_upload_bytes_float32=int(plain_bytes), ratio_vs_plain=round(led.total_bytes() / plain_bytes, 2),
            ratio_vs_plain_excluding_placeholder=round((led.total_bytes() - placeholder_cmp) / plain_bytes, 2),
            offline_triple_bytes=int(led.bytes_.get("offline", 0)))
json.dump(cost, open(OUT_DIR / "cost.json", "w"), indent=2); print(json.dumps(cost, indent=1))

## 7. Export (aggregates only, no raw data)

In [ ]:
# ============================== 7. EXPORT ==============================
audit_info = dict(file_rows=FILE_ROWS, file_class_counts=FILE_COUNTS, classes=CLASSES, sampled_rows=int(len(y_arr)),
                  sampled_attack_share=float(y_bin.mean()), sampled_class_counts=pd.Series(types_arr).value_counts().to_dict(),
                  n_features=len(FEATURES), feature_names=FEATURES, feature_groups_before_selection=groups,
                  skipped_columns=skipped, model_dim=D, kappa=KAPPA, partitions=bundle_info)
json.dump(audit_info, open(OUT_DIR / "data_audit.json", "w"), indent=1)
env = dict(python=sys.version, platform=platform.platform(), numpy=np.__version__, pandas=pd.__version__,
           cpu_count=os.cpu_count(), finished=datetime.datetime.now().isoformat(timespec="seconds"))
json.dump(env, open(OUT_DIR / "env.json", "w"), indent=1)
zip_path = OUT_DIR / "results_pack.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["config.json", "results.csv", "summary.csv", "paired.csv", "fairness.csv", "cost.json", "data_audit.json", "env.json"]:
        if (OUT_DIR / name).exists(): z.write(OUT_DIR / name, name)
    for p in (OUT_DIR / "figs").glob("*.png"): z.write(p, f"figs/{p.name}")
print(f"wrote {zip_path.resolve()} ({zip_path.stat().st_size / 1024:.0f} KiB). Upload it to the chat (aggregates only).")